In [0]:
%pip install databricks_langchain==0.9.0 -q
%pip install databricks-vectorsearch -q
%pip install mlflow-skinny[databricks] -q
%pip install langchain==1.0.0 -q
%pip install langgraph==1.0.10 -q

dbutils.library.restartPython()

In [0]:
import json
import mlflow
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.state import CompiledStateGraph

mlflow.langchain.autolog()
LLM_ENDPOINT_NAME = "databricks-gpt-oss-120b"
INDEX_NAME = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_chunked_index"

In [0]:
def build_agent(llm_endpoint: str, index_name: str, num_results: int = 3) -> CompiledStateGraph:
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=500
    )

    vs_tool = VectorSearchRetrieverTool(
        index_name=index_name,
        description="Sephora pricing strategy",
        num_results=num_results,
    )

    checkpointer = InMemorySaver()

    system_prompt = """You are expert on Sephora's pricing strategy. Respond in a clear, professional, and factual tone
    appropriate for engineers and technical staff. Use only verified information from the provided context, include 
    reference to every information you provide. Do not hallucinate information. If you are unsure about the answer,
    say "I don't know"."""

    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

    return agent

agent = build_agent(
    llm_endpoint=LLM_ENDPOINT_NAME,
    index_name=INDEX_NAME,
    num_results=3
)

In [0]:
config = {"configurable": {"thread_id": "sephora_agent_test"}}
response = agent.invoke(
    input={
        "messages": [
            {
                "role": "user",
                "content": "What is the pricing strategy for Sephora's skincare products"
            }
        ]
    },
    config=config
)

# response_str = json.dumps(response["messages"][-1].content, indent=4)
print(response["messages"][-1].content)

**Sephora’s pricing strategy for its skincare portfolio is a data‑driven, algorithmic approach that combines real‑time competitive intelligence with historical‑context analytics to support automated, profit‑focused price‑optimization across multiple markets.**  

| Component | How it is implemented | Evidence |
|-----------|----------------------|----------|
| **Centralised competitive‑price collection** | A single platform continuously scrapes competitor **shelf‑price** and **coupon‑price** data from 35 competitor websites and 4 marketplaces. The feed delivers >5.9 million data points per day, giving Sephora an up‑to‑the‑minute view of how rivals price comparable skincare SKUs. | “Daily processing exceeding 5.9 million data points… Monitoring expanded to 35 competitor websites and 4 marketplaces” (Document ID 3). |
| **Historical, ML‑ready dataset** | The pipeline stores not only current prices but also product‑stock periods, delivery terms, and brand‑availability patterns, creating a rich historical context that can be fed directly into machine‑learning models. | “ML‑ready data with historical context (e.g., product stock periods, delivery terms, brand availability patterns at competitor retailers)” (Document ID 2). |
| **High‑quality data assurance** | The proof‑of‑concept achieved **98 % match quality** and **99 % data‑accuracy**, providing a reliable foundation for pricing algorithms. | “Achieved 98 % match quality and 99 % data accuracy” (Document ID 2). |
| **Advanced pricing algorithms & localized strategies** | Using the clean data set, Sephora runs sophisticated pricing algorithms that balance two core objectives: <br>1. **Maximise profitability** (margin‑focused price adjustments). <br>

In [0]:
import yaml

def create_agent_config(llm_endpoint_name: str, index_name: str, system_prompt: str, num_results: int = 3) -> dict:
    config = {
        "llm_endpoint_name": llm_endpoint_name,
        'vector_search':{
            "index_name": index_name,
            "num_results": num_results,
            "system_propmt": system_prompt
        },
        
    }
    return config


# Create config file

agent_config = create_agent_config(LLM_ENDPOINT_NAME, INDEX_NAME, system_prompt)

config_yaml = yaml.safe_dump(agent_config, sort_keys=False)

with open('agent_config.yaml', 'w', encoding='utf-8') as file:
    file.write(config_yaml)

print(config_yaml)

In [0]:
import os
from uuid import uuid4
from typing import Any

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.state import CompiledStateGraph


def _load_config(path: str = "agent_config.yaml") -> dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at path: {path}")
    with open(path, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f) or {}
    
    llm_endpoint = cfg["llm_endpoint"]
    index_name = vs["index_name"]
    system_prompt = cfg["system_prompt"]

    vs = cfg.get("vector_search", {}) or {}
    num_results = int(vs.get("num_results", 5))
    
    return {
        "llm_endpoint": llm_endpoint,
        "vs_index_name": index_name,
        "vs_num_results": num_results,
        "vs_system_prompt": system_prompt
    }


def build_agent(llm_endpoint: str, index_name: str, system_prompt: str, num_results: int = 3) -> CompiledStateGraph:
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=500
    )

    vs_tool = VectorSearchRetrieverTool(
        index_name=index_name,
        description="Sephora pricing strategy",
        num_results=num_results,
    )

    checkpointer = InMemorySaver()

    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

    return agent